In [24]:
import requests
import json
import pandas as pd
import csv
import psycopg2


In [25]:
# Read into a DataFrame
propertyRecords_df = pd.read_json('C:\\Users\\USER\\Desktop\\Projects\\PERSONAL PROJECTS\\FREE PROJECTS\\Real Estate Data Evolution\\PropertyRecords.json')


In [26]:
propertyRecords_df.head()

,addressLine1,city,state,zipCode,formattedAddress,assessorID,county,legalDescription,ownerOccupied,squareFootage,...,bathrooms,features,taxAssessment,propertyTaxes,owner,id,longitude,latitude,bedrooms,addressLine2
0,30631 W Celeborn Dr,Buckeye,AZ,85396,"30631 W Celeborn Dr, Buckeye, AZ 85396",504-75-191,Maricopa,TARTESSO UNIT 2A MCR 754-28,1.0,2025.0,...,3.0,"{'exteriorType': 'Stucco', 'floorCount': 1, 'g...","{'2022': {'value': 2420, 'land': 2420}, '2023'...","{'2021': {'total': 125}, '2022': {'total': 1593}}","{'names': ['ADRIAN RAMIREZ', 'MONICA JOANNA LO...","30631-W-Celeborn-Dr,-Buckeye,-AZ-85396",-112.710832,33.481631,NaN,NaN
1,1212 Horseman Pl,Bismarck,ND,58501,"1212 Horseman Pl, Bismarck, ND 58501",1389-003-035,Burleigh,SLEEPY HOLLOW HEIGHTS 5TH LOT 8,1.0,2306.0,...,3.0,"{'architectureType': 'French Provincial', 'coo...","{'2022': {'value': 184550, 'land': 37000, 'imp...",{'2022': {'total': 5158}},"{'names': ['ZACHARY P BOEHLER'], 'mailingAddre...","1212-Horseman-Pl,-Bismarck,-ND-58501",-100.746177,46.818761,4.0,NaN
2,11030 Creekbridge Pl,San Diego,CA,92128,"11030 Creekbridge Pl, San Diego, CA 92128",316-232-07-09,San Diego,PM15983 PAR 2*US 74PER DOC90-202308&UND INT IN,NaN,1273.0,...,2.5,"{'architectureType': 'Condo / Apartment', 'gar...","{'2020': {'value': 329281, 'land': 151552, 'im...","{'2020': {'total': 3559}, '2022': {'total': 36...","{'names': ['INGRID E ENSCHSIMON'], 'mailingAdd...","11030-Creekbridge-Pl,-San-Diego,-CA-92128",-117.092509,32.945955,2.0,NaN
3,1113 W High St,Haddon Heights,NJ,8035,"1113 W High St, Haddon Heights, NJ 08035",NaN,Camden County,NaN,NaN,1936.0,...,1.5,NaN,NaN,NaN,NaN,"1113-W-High-St,-Haddon-Heights,-NJ-08035",-75.067199,39.875404,3.0,NaN
4,441 Janice Kay Pl,Kissimmee,FL,34744,"441 Janice Kay Pl, Kissimmee, FL 34744",292530299600010780,Osceola,EAGLES LANDING PB 15 PGS 121-122 LOT 78,1.0,2620.0,...,2.0,"{'cooling': True, 'coolingType': 'Package', 'e...",{'2022': {'value': 136633}},{'2022': {'total': 2823}},"{'names': ['MEADEMATIAS JESSICA', 'MATIAS REIN...","441-Janice-Kay-Pl,-Kissimmee,-FL-34744",-81.335555,28.287159,3.0,NaN


In [27]:
# Dataset information
propertyRecords_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   addressLine1      500 non-null    object 
 1   city              500 non-null    object 
 2   state             500 non-null    object 
 3   zipCode           500 non-null    int64  
 4   formattedAddress  500 non-null    object 
 5   assessorID        338 non-null    object 
 6   county            500 non-null    object 
 7   legalDescription  327 non-null    object 
 8   ownerOccupied     255 non-null    float64
 9   squareFootage     389 non-null    float64
 10  subdivision       289 non-null    object 
 11  yearBuilt         343 non-null    float64
 12  zoning            179 non-null    object 
 13  lotSize           330 non-null    float64
 14  propertyType      418 non-null    object 
 15  lastSalePrice     229 non-null    float64
 16  lastSaleDate      289 non-null    object 
 1

In [28]:
# Fill missing values with defaults or placeholders
propertyRecords_df.fillna({
    'assessorID': 'Unknown',
    'legalDescription': 'Not available',
    'ownerOccupied': 0,
    'squareFootage': 0,
    'subdivision': 'Not available',
    'yearBuilt': 0,
    'zoning' : 'Unknown',
    'lotSize': 0,
    'propertyType': 'Unknown',
    'lastSalePrice': 0,
    'lastSaleDate': 'Not available',
    'bathrooms': 0,
    'taxAssessment': 'Not available',
    'propertyTaxes': 'Not available',
    'owner': 'Unknown',
    'bedrooms': 0,
    'addressLine2': 'Not available',

   }, inplace=True)

Transformation Layer

In [29]:
# 1st Convert dictionary column to string
propertyRecords_df['features'] = propertyRecords_df['features'].apply(json.dumps)

In [30]:
# 2nd step replace NAN values with appropriate defaults or remove row/columns as necessary
propertyRecords_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   addressLine1      500 non-null    object 
 1   city              500 non-null    object 
 2   state             500 non-null    object 
 3   zipCode           500 non-null    int64  
 4   formattedAddress  500 non-null    object 
 5   assessorID        500 non-null    object 
 6   county            500 non-null    object 
 7   legalDescription  500 non-null    object 
 8   ownerOccupied     500 non-null    float64
 9   squareFootage     500 non-null    float64
 10  subdivision       500 non-null    object 
 11  yearBuilt         500 non-null    float64
 12  zoning            500 non-null    object 
 13  lotSize           500 non-null    float64
 14  propertyType      500 non-null    object 
 15  lastSalePrice     500 non-null    float64
 16  lastSaleDate      500 non-null    object 
 1

In [31]:
# Create the fact table
fact_columns = ['addressLine1', 'city', 'state', 'zipCode', 'formattedAddress', 'squareFootage', 'yearBuilt',
               'bedrooms','bathrooms', 'lotSize', 'propertyType', 'longitude', 'latitude']
fact_table = propertyRecords_df[fact_columns]
fact_table. head()


,addressLine1,city,state,zipCode,formattedAddress,squareFootage,yearBuilt,bedrooms,bathrooms,lotSize,propertyType,longitude,latitude
0,30631 W Celeborn Dr,Buckeye,AZ,85396,"30631 W Celeborn Dr, Buckeye, AZ 85396",2025.0,2021.0,0.0,3.0,6960.0,Single Family,-112.710832,33.481631
1,1212 Horseman Pl,Bismarck,ND,58501,"1212 Horseman Pl, Bismarck, ND 58501",2306.0,2013.0,4.0,3.0,20491.0,Single Family,-100.746177,46.818761
2,11030 Creekbridge Pl,San Diego,CA,92128,"11030 Creekbridge Pl, San Diego, CA 92128",1273.0,2000.0,2.0,2.5,145899.0,Condo,-117.092509,32.945955
3,1113 W High St,Haddon Heights,NJ,8035,"1113 W High St, Haddon Heights, NJ 08035",1936.0,0.0,3.0,1.5,0.0,Single Family,-75.067199,39.875404
4,441 Janice Kay Pl,Kissimmee,FL,34744,"441 Janice Kay Pl, Kissimmee, FL 34744",2620.0,2005.0,3.0,2.0,5489.0,Single Family,-81.335555,28.287159


In [32]:
# Create Location Dimension
location_dim = propertyRecords_df[['addressLine1', 'city', 'state', 'zipCode', 'county', 'longitude', 'latitude']].drop_duplicates().reset_index(drop=True)
location_dim.index.name = 'location_id'
location_dim. head()

,addressLine1,city,state,zipCode,county,longitude,latitude
location_id,,,,,,,
0,30631 W Celeborn Dr,Buckeye,AZ,85396,Maricopa,-112.710832,33.481631
1,1212 Horseman Pl,Bismarck,ND,58501,Burleigh,-100.746177,46.818761
2,11030 Creekbridge Pl,San Diego,CA,92128,San Diego,-117.092509,32.945955
3,1113 W High St,Haddon Heights,NJ,8035,Camden County,-75.067199,39.875404
4,441 Janice Kay Pl,Kissimmee,FL,34744,Osceola,-81.335555,28.287159


In [33]:
# 2nd step replace NAN values with appropriate defaults or remove row/columns as necessary
propertyRecords_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   addressLine1      500 non-null    object 
 1   city              500 non-null    object 
 2   state             500 non-null    object 
 3   zipCode           500 non-null    int64  
 4   formattedAddress  500 non-null    object 
 5   assessorID        500 non-null    object 
 6   county            500 non-null    object 
 7   legalDescription  500 non-null    object 
 8   ownerOccupied     500 non-null    float64
 9   squareFootage     500 non-null    float64
 10  subdivision       500 non-null    object 
 11  yearBuilt         500 non-null    float64
 12  zoning            500 non-null    object 
 13  lotSize           500 non-null    float64
 14  propertyType      500 non-null    object 
 15  lastSalePrice     500 non-null    float64
 16  lastSaleDate      500 non-null    object 
 1

In [34]:
# Create Sales Dimension
sales_dim = propertyRecords_df[['lastSalePrice', 'lastSaleDate']].drop_duplicates().reset_index(drop=True)
sales_dim.index.name = 'sales_id'
sales_dim. head()

,lastSalePrice,lastSaleDate
sales_id,,
0,325990.0,2021-09-02T00:00:00.000Z
1,330000.0,2014-10-03T00:00:00.000Z
2,239000.0,2012-03-27T00:00:00.000Z
3,0.0,Not available
4,217800.0,2005-07-18T00:00:00.000Z


In [35]:
# Create Property Features Dimension
features_dim = propertyRecords_df[['features', 'propertyType', 'zoning']].drop_duplicates().reset_index(drop=True)
features_dim.index.name = 'features_id'
features_dim. head()

,features,propertyType,zoning
features_id,,,
0,"{""exteriorType"": ""Stucco"", ""floorCount"": 1, ""g...",Single Family,PAD
1,"{""architectureType"": ""French Provincial"", ""coo...",Single Family,R5
2,"{""architectureType"": ""Condo / Apartment"", ""gar...",Condo,R-1:SINGLE FAM-RES
3,NaN,Single Family,Unknown
4,"{""cooling"": true, ""coolingType"": ""Package"", ""e...",Single Family,OPUD


In [36]:

fact_table.to_csv('property_fact.csv', index=False)
location_dim.to_csv('location_dimension.csv', index=True)
sales_dim.to_csv('sales_dimension.csv', index=True)
features_dim.to_csv('features_dimension.csv', index=True)


In [37]:
features_dim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 333 entries, 0 to 332
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   features      333 non-null    object
 1   propertyType  333 non-null    object
 2   zoning        333 non-null    object
dtypes: object(3)
memory usage: 7.9+ KB


Loading Layer

In [38]:
# Develop a function to connect to pgadmin
def get_db_connection():
  connection = psycopg2.connect(
      host="localhost",
      port=5432,
      dbname="postgres",
      user="postgres",
      password="admin456"
  )
  return connection

  conn = get_db_connection()

In [39]:
# Create tables
def create_tables():
  conn = get_db_connection()
  cursor = conn.cursor()
  create_table_query = '''-- Drop existing tables
                          DROP TABLE IF EXISTS Zapbank.fact_table;
                          DROP TABLE IF EXISTS Zapbank.location_dim;
                          DROP TABLE IF EXISTS Zapbank.sales_dim;
                          DROP TABLE IF EXISTS Zapbank.features_dim;

                           -- Create the schema if it doesn't exist
                          CREATE SCHEMA IF NOT EXISTS Zapbank;
                         
                          -- Create new tables
                          CREATE TABLE Zapbank.fact_table (
                            addressLine1 VARCHAR(255),
                            city VARCHAR(100),
                            state VARCHAR(50),
                            zipCode INTEGER,
                            formattedAddress VARCHAR(255),
                            squareFootage FLOAT,
                            yearBuilt FLOAT,
                            bedrooms FLOAT,
                            bathrooms FLOAT,
                            lotSize FLOAT,
                            propertyType VARCHAR(100),
                            longitude FLOAT,
                            latitude FLOAT
                          );

                          CREATE TABLE Zapbank.location_dim (
                            location_id SERIAL PRIMARY KEY,
                            addressLine1 VARCHAR(255),
                            city VARCHAR (100),
                            state VARCHAR(50),
                            zipCode INTEGER,
                            county VARCHAR(100),
                            longitude FLOAT,
                            latitude FLOAT
                          );

                          CREATE TABLE Zapbank.sales_dim (
                            sales_id SERIAL PRIMARY KEY,
                            lastSalePrice FLOAT,
                            lastSaleDate DATE
                          );

                          CREATE TABLE Zapbank.features_dim(
                            features_id SERIAL PRIMARY KEY,
                            features TEXT,
                            propertyType VARCHAR(100),
                            zoning VARCHAR(100)
                          );'''
  
  cursor.execute(create_table_query)
  conn.commit()
  cursor.close()
  conn.close()

create_tables()


In [40]:
# Create a function to load the csv data into the database
def load_data_from_csv_to_table(csv_path, table_name):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader)  # Skip the header row
        for row in reader:
            placeholders = ', '.join(['%s'] * len(row))
            query = f'INSERT INTO {table_name} VALUES ({placeholders});'
            cursor.execute(query, row)

    conn.commit()
    cursor.close()
    conn.close()


In [41]:
# Create a function to load the csv data into the database
def load_data_from_csv_to_table(csv_path, table_name):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader)  # Skip the header row
        for row in reader:
            placeholders = ', '.join(['%s'] * len(row))
            query = f'INSERT INTO {table_name} VALUES ({placeholders});'
            cursor.execute(query, row)
    conn.commit()
    cursor.close()
    conn.close()


In [42]:
# fact table
fact_csv_path = (r'C:\Users\USER\Desktop\Projects\PERSONAL PROJECTS\FREE PROJECTS\Real Estate Data Evolution\property_fact.csv')
load_data_from_csv_to_table(fact_csv_path, 'Zapbank.fact_table')


In [43]:
# location dimension table
location_csv_path = (r'C:\Users\USER\Desktop\Projects\PERSONAL PROJECTS\FREE PROJECTS\Real Estate Data Evolution\location_dimension.csv')
load_data_from_csv_to_table(location_csv_path, 'Zapbank.location_dim')


In [44]:
# features table
features_csv_path = (r'C:\Users\USER\Desktop\Projects\PERSONAL PROJECTS\FREE PROJECTS\Real Estate Data Evolution\features_dimension.csv')
load_data_from_csv_to_table(features_csv_path, 'Zapbank.features_dim')


In [45]:
def load_data_from_csv_to_sales_table(csv_path, table_name):
    conn = get_db_connection()
    cursor = conn.cursor()
    
    with open(csv_path, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader)  # Skip the header row
        
        for row in reader:
            # Convert empty strings (or 'Not available') in date column to None (NULL in SQL)
            row = [None if (cell == '' or cell == 'Not available') and col_name == 'lastSaleDate' else cell for cell, col_name in zip(row, sales_dim_columns)]

            placeholders = ', '.join(['%s'] * len(row))
            query = f'INSERT INTO {table_name} VALUES ({placeholders});'
            cursor.execute(query, row)

    conn.commit()
    cursor.close()
    conn.close()

# define the columns names in sales_dim table
sales_dim_columns = ['sales_id', 'lastSalePrice', 'lastSaleDate']

# features table
sales_csv_path = (r'C:\Users\USER\Desktop\Projects\PERSONAL PROJECTS\FREE PROJECTS\Real Estate Data Evolution\sales_dimension.csv')
load_data_from_csv_to_sales_table(sales_csv_path, 'Zapbank.sales_dim')

          

In [46]:
print('All Data has been loaded successfully into their respective schema and tables')


All Data has been loaded successfully into their respective schema and tables
